# Práctica guiada · Ajuste de hiperparámetros con tres clasificadores
### Regresión logística · SVM · Naive Bayes

**FP13 · Inteligencia Artificial** —  Universidad Autónoma de Guadalajara

**Unidad 5:** 5.5 Regresión logística · 5.6 Naive Bayes · 5.7 Matrices de confusión · 5.8 SVM — con adelanto de **6.5 Búsqueda de rejilla**

---

### 🏥 El problema clínico
Un servicio de cardiología quiere una herramienta de **apoyo al triaje**: con 11 datos clínicos de rutina (edad, tipo de dolor torácico, colesterol, ECG en reposo, frecuencia cardiaca máxima, depresión del segmento ST…) debe decidir si un paciente **probablemente tiene enfermedad cardiaca** y conviene enviarlo a estudios especializados.

In [1]:
import warnings
warnings.filterwarnings("ignore")      # oculta avisos técnicos para que la pantalla quede limpia

import numpy as np                     # cálculo numérico
import pandas as pd                    # tablas de datos
import matplotlib.pyplot as plt        # gráficas
import kagglehub                       # descarga datasets de Kaggle

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay, confusion_matrix,
                             accuracy_score, recall_score, precision_score, f1_score)

---
## 1 · Los datos

**Heart Failure Prediction Dataset** (fedesoriano, Kaggle): 918 pacientes de cinco cohortes cardiológicas.

| Variable | Significado clínico |
|---|---|
| `Age` | Edad (años) |
| `Sex` | Sexo (M / F) |
| `ChestPainType` | Dolor torácico: TA angina típica · ATA atípica · NAP no anginoso · ASY asintomático |
| `RestingBP` | Presión arterial sistólica en reposo (mm Hg) |
| `Cholesterol` | Colesterol sérico (mg/dL) |
| `FastingBS` | Glucosa en ayuno > 120 mg/dL (1 = sí) |
| `RestingECG` | ECG en reposo: Normal · ST (anomalía ST-T) · LVH (hipertrofia ventricular izquierda) |
| `MaxHR` | Frecuencia cardiaca máxima en prueba de esfuerzo |
| `ExerciseAngina` | Angina inducida por el ejercicio (Y / N) |
| `Oldpeak` | Depresión del segmento ST con el esfuerzo (mm) |
| `ST_Slope` | Pendiente del ST en el pico del esfuerzo: Up · Flat · Down |
| **`HeartDisease`** | **Lo que queremos predecir: 1 = con enfermedad cardiaca · 0 = sin enfermedad** |

In [ ]:
# Importando  datos
ruta = kagglehub.dataset_download('fedesoriano/heart-failure-prediction')

df = pd.read_csv(ruta + '/heart.csv')

100%|██████████| 8.56k/8.56k [00:00<00:00, 2.74MB/s]

Extracting files...


In [ ]:
# Explorando datos
df.head()


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [9]:
# Imprimiendo renglones y columnas
df.shape

(918, 12)

---
## 2 · Preparar los datos

Tres decisiones, todas vistas en la Unidad 4:
1. **Valores imposibles → faltantes.** Nadie vivo tiene colesterol 0 ni presión arterial 0: son celdas que nadie llenó.
2. **Separar `X` de `y`.** `X` = lo que el modelo puede ver; `y` = la respuesta correcta.
3. **One-hot.** Los modelos solo entienden números, así que cada columna de texto se convierte en columnas de 0 y 1.

In [13]:
# Imprimiendo tipos de datos
df.dtypes

Age                 int64
Sex                   str
ChestPainType         str
RestingBP           int64
Cholesterol         int64
FastingBS           int64
RestingECG            str
MaxHR               int64
ExerciseAngina        str
Oldpeak           float64
ST_Slope              str
HeartDisease        int64
dtype: object

In [14]:
df.isna().sum()

Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64

In [15]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,918.0,53.510893,9.432617,28.0,47.00,54.0,60.0,77.0
RestingBP,918.0,132.396514,18.514154,0.0,120.00,130.0,140.0,200.0
Cholesterol,918.0,198.799564,109.384145,0.0,173.25,223.0,267.0,603.0
FastingBS,918.0,0.233115,0.423046,0.0,0.00,0.0,0.0,1.0
MaxHR,918.0,136.809368,25.460334,60.0,120.00,138.0,156.0,202.0
Oldpeak,918.0,0.887364,1.066570,-2.6,0.00,0.6,1.5,6.2
HeartDisease,918.0,0.553377,0.497414,0.0,0.00,1.0,1.0,1.0


In [ ]:
# Corrigiendo valores numericos en 0
df['Cholesterol'] = df['Cholesterol'].replace(0, np.nan)
df['RestingBP'] = df['RestingBP'].replace(0, np.nan)



In [17]:
df.isna().sum()

Age                 0
Sex                 0
ChestPainType       0
RestingBP           1
Cholesterol       172
FastingBS           0
RestingECG          0
MaxHR               0
ExerciseAngina      0
Oldpeak             0
ST_Slope            0
HeartDisease        0
dtype: int64

# Dividiendo los datos y aplicando one hot

In [18]:
# Separando datos en x e y
y = df['HeartDisease']
X = df.drop(columns=['HeartDisease'])



In [21]:
string_colums = ['Sex','ChestPainType','RestingECG','ExerciseAngina','ST_Slope']
for colum in string_colums:
    print(X[colum].value_counts())
    print('\n')

Sex
M    725
F    193
Name: count, dtype: int64


ChestPainType
ASY    496
NAP    203
ATA    173
TA      46
Name: count, dtype: int64


RestingECG
Normal    552
LVH       188
ST        178
Name: count, dtype: int64


ExerciseAngina
N    547
Y    371
Name: count, dtype: int64


ST_Slope
Flat    460
Up      395
Down     63
Name: count, dtype: int64




In [22]:
#Imprimiendo datos antes del one hot
print(X.head())
print(X.shape)


   Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  MaxHR  \
0   40   M           ATA      140.0        289.0          0     Normal    172   
1   49   F           NAP      160.0        180.0          0     Normal    156   
2   37   M           ATA      130.0        283.0          0         ST     98   
3   48   F           ASY      138.0        214.0          0     Normal    108   
4   54   M           NAP      150.0        195.0          0     Normal    122   

  ExerciseAngina  Oldpeak ST_Slope  
0              N      0.0       Up  
1              N      1.0     Flat  
2              N      0.0       Up  
3              Y      1.5     Flat  
4              N      0.0       Up  
(918, 11)


In [24]:
# Aplicando one hot
X = pd.get_dummies(X, drop_first=True, dtype= int)

print(X.head())
print(X.shape)

   Age  RestingBP  Cholesterol  FastingBS  MaxHR  Oldpeak  Sex_M  \
0   40      140.0        289.0          0    172      0.0      1   
1   49      160.0        180.0          0    156      1.0      0   
2   37      130.0        283.0          0     98      0.0      1   
3   48      138.0        214.0          0    108      1.5      0   
4   54      150.0        195.0          0    122      0.0      1   

   ChestPainType_ATA  ChestPainType_NAP  ChestPainType_TA  RestingECG_Normal  \
0                  1                  0                 0                  1   
1                  0                  1                 0                  1   
2                  1                  0                 0                  0   
3                  0                  0                 0                  1   
4                  0                  1                 0                  1   

   RestingECG_ST  ExerciseAngina_Y  ST_Slope_Flat  ST_Slope_Up  
0              0                 0           

# Haciendo el split de los datos

In [28]:
# Imprimiendo porcion de la variable objetivo
df['HeartDisease'].value_counts(normalize=True)

proporcion = df['HeartDisease'].value_counts(normalize=True)[1]
print('\n')
print(f'Los datos estan casi balanceados ({proporcion:.1%}), pero aun asi se estratificaran en el split de datos')




Los datos estan casi balanceados (55.3%), pero aun asi se estratificaran en el split de datos


In [30]:
# Dividiendo los datos en entrenamiento y prueba
X_train, X_test, y_train, y_text = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

print(f'Dimenciones de entrenamiento del conjunto X: {X_train.shape}')
print(f'Dimenciones de prueba del conjunto X: {X_test.shape}')

Dimenciones de entrenamiento del conjunto X: (734, 15)
Dimenciones de prueba del conjunto X: (184, 15)


---
## 3 · Repaso: ¿cómo vamos a evaluar?

Las filas son la **realidad** y las columnas lo que **predice** el modelo:

| | Predice **Sano** | Predice **Enfermo** |
|---|:---:|:---:|
| **Realmente sano** | ✅ VN · sano bien descartado | ⚠️ FP · falsa alarma |
| **Realmente enfermo** | ❌ **FN · enfermo que se va a casa** | ✅ VP · enfermo detectado |

<br>

| Métrica | Pregunta clínica que responde | Nombre en `scoring` |
|---|---|---|
| **Exactitud** | De cada 100 pacientes, ¿cuántos clasifica bien? | `"accuracy"` |
| **Precisión** | De cada 100 que el modelo marca como enfermos, ¿cuántos realmente lo están? | `"precision"` |
| **Sensibilidad** | De cada 100 enfermos, ¿a cuántos detecta? | `"recall"` |
| **F1** | Un solo número que exige **a la vez** buena precisión y buena sensibilidad | `"f1"` |
| **AUC** | Si tomo un enfermo y un sano al azar, ¿qué tan seguido el modelo le asigna más riesgo al enfermo? | `"roc_auc"` |
| **Especificidad** | De cada 100 sanos, ¿a cuántos descarta correctamente? | *(es el recall de la clase "Sano")* |

---
# Regresion logistica

In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer

columnas_escalar = ['Age','RestingBP','Cholesterol','MaxHR','Oldpeak']

prep = ColumnTransformer(
    transformers=[("escalar", StandardScaler(),columnas_escalar)],
    remainder="passtrough",
    verbose_feature_names_out=False
)

reglog_base = Pipeline([
    ("imputar", SimpleImputer(strategy="median")),
    ("escalar", StandardScaler()),
    ("modelo", LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0,
                                  max_iter=5000, random_state=42))

])


reglog_base.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputar', ...), ('escalar', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](15,)","['Age','RestingBP','Cholesterol',...,'ExerciseAngina_Y','ST_Slope_Flat', 'ST_Slope_Up']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,15
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. Fo

In [ ]:
columnas_escalar = ['Age','RestingBP','Cholesterol','MaxHR','Oldpeak']
